<a href="https://colab.research.google.com/github/mmilannaik/bostonhousepricing/blob/main/W16S1_SQL_Window_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Install pandasql
!pip install pandasql --quiet

# 2. Load libraries and your CSV into a pandas DataFrame
import pandas as pd
from pandasql import sqldf

# 3. Create a helper to run SQL against any DataFrame in your notebook
pysqldf = lambda query: sqldf(query, globals())

  Preparing metadata (setup.py) ... done


In [ ]:
marks = pd.read_csv('/content/W16S1_student_marks.csv')
marks.head(2)

,name,branch,marks
0,Nitish,EEE,82
1,Rishabh,EEE,91


In [ ]:
pysqldf('''
  SELECT *,AVG(marks) OVER(PARTITION by branch) AS 'branch_avg'
  FROM marks

''')

,name,branch,marks,branch_avg
0,Shubham,CSE,78,78.50
1,Ved,CSE,43,78.50
2,Deepak,CSE,98,78.50
3,Arpan,CSE,95,78.50
4,Vinay,ECE,95,89.75
5,Ankit,ECE,88,89.75
6,Anand,ECE,81,89.75
7,Rohit,ECE,95,89.75
8,Nitish,EEE,82,74.25
9,Rishabh,EEE,91,74.25


# Q1 :Aggregate Function with OVER

In [ ]:
pysqldf('''
SELECT *,
MIN(marks) OVER(),
MAX(marks) OVER()
FROM marks
''')

,name,branch,marks,MIN(marks) OVER(),MAX(marks) OVER()
0,Nitish,EEE,82,39,98
1,Rishabh,EEE,91,39,98
2,Anukant,EEE,69,39,98
3,Rupesh,EEE,55,39,98
4,Shubham,CSE,78,39,98
5,Ved,CSE,43,39,98
6,Deepak,CSE,98,39,98
7,Arpan,CSE,95,39,98
8,Vinay,ECE,95,39,98
9,Ankit,ECE,88,39,98


# Q2: Find all students who have marks higher than avg marks of their respective branch

In [ ]:
pysqldf('''
SELECT  * FROM (SELECT *, AVG(marks) OVER(PARTITION by branch) AS 'branch_avg'
FROM marks) t
WHERE t.marks > t.branch_avg
''')

,name,branch,marks,branch_avg
0,Deepak,CSE,98,78.50
1,Arpan,CSE,95,78.50
2,Vinay,ECE,95,89.75
3,Rohit,ECE,95,89.75
4,Nitish,EEE,82,74.25
5,Rishabh,EEE,91,74.25
6,Prashant,MECH,75,58.50
7,Amit,MECH,69,58.50


# Q3: Rank

In [ ]:
pysqldf('''
SELECT * ,
  rank() over(partition by branch order by marks desc)
  from marks


''')

,name,branch,marks,rank() over(partition by branch order by marks desc)
0,Deepak,CSE,98,1
1,Arpan,CSE,95,2
2,Shubham,CSE,78,3
3,Ved,CSE,43,4
4,Vinay,ECE,95,1
5,Rohit,ECE,95,1
6,Ankit,ECE,88,3
7,Anand,ECE,81,4
8,Rishabh,EEE,91,1
9,Nitish,EEE,82,2


# Q4. DENSE_RANK

In [ ]:
pysqldf('''
SELECT * ,
  rank() over(partition by branch order by marks desc),
  dense_rank() over(partition by branch order by marks desc)
  from marks


''')

,name,branch,marks,rank() over(partition by branch order by marks desc),dense_rank() over(partition by branch order by marks desc)
0,Deepak,CSE,98,1,1
1,Arpan,CSE,95,2,2
2,Shubham,CSE,78,3,3
3,Ved,CSE,43,4,4
4,Vinay,ECE,95,1,1
5,Rohit,ECE,95,1,1
6,Ankit,ECE,88,3,2
7,Anand,ECE,81,4,3
8,Rishabh,EEE,91,1,1
9,Nitish,EEE,82,2,2


In [ ]:
pysqldf('''
SELECT * ,
  CONCAT(branch,'-',row_number() over(partition by branch))
  from marks


''')

PandaSQLException: (sqlite3.OperationalError) no such function: CONCAT
[SQL: 
SELECT * ,
  CONCAT(branch,'-',row_number() over(partition by branch))
  from marks


]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [3]:
orders = pd.read_csv('/content/W15S2_orders.csv')
orders.head(2)

,order_id,user_id,r_id,amount,date,partner_id,delivery_time,delivery_rating,restaurant_rating
0,1001,1,1,550,2022-05-10,1,25,5,3.0
1,1002,1,2,415,2022-05-26,1,19,5,2.0


# Q5: Top 2 paying customers each month

In [ ]:
pysqldf('''
SELECT * FROM
(SELECT strftime('%m', date) AS month, user_id,SUM(amount) AS 'total',
RANK() OVER(PARTITION BY strftime('%m', date)  ORDER BY SUM(amount) DESC) AS 'month_rank'
  from orders
  GROUP BY strftime('%m', date),user_id
  ORDER BY strftime('%m', date)) t
  WHERE t.month_rank < 3
  ORDER BY month desc,month_rank ASC

''')

,month,user_id,total,month_rank
0,07,5,3035,1
1,07,2,1190,2
2,06,2,1480,1
3,06,4,800,2
4,05,1,965,1
5,05,3,860,2


# Q6: Find Branch Toppers

In [ ]:
pysqldf('''
SELECT *,
FIRST_VALUE(marks) OVER(ORDER BY marks DESC) FROM marks

''')

,name,branch,marks,FIRST_VALUE(marks) OVER(ORDER BY marks DESC)
0,Deepak,CSE,98,98
1,Arpan,CSE,95,98
2,Vinay,ECE,95,98
3,Rohit,ECE,95,98
4,Rishabh,EEE,91,98
5,Ankit,ECE,88,98
6,Nitish,EEE,82,98
7,Anand,ECE,81,98
8,Shubham,CSE,78,98
9,Prashant,MECH,75,98


In [ ]:
pysqldf('''
SELECT *,
FIRST_VALUE(name) OVER(ORDER BY marks DESC) FROM marks

''')

,name,branch,marks,FIRST_VALUE(name) OVER(ORDER BY marks DESC)
0,Deepak,CSE,98,Deepak
1,Arpan,CSE,95,Deepak
2,Vinay,ECE,95,Deepak
3,Rohit,ECE,95,Deepak
4,Rishabh,EEE,91,Deepak
5,Ankit,ECE,88,Deepak
6,Nitish,EEE,82,Deepak
7,Anand,ECE,81,Deepak
8,Shubham,CSE,78,Deepak
9,Prashant,MECH,75,Deepak


In [ ]:
pysqldf('''
SELECT *,
last_value(marks) OVER(PARTITION BY branch
                      ORDER BY marks DESC
                  ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) FROM marks

''')

,name,branch,marks,last_value(marks) OVER(PARTITION BY branch\n ORDER BY marks DESC\n ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)
0,Deepak,CSE,98,43
1,Arpan,CSE,95,43
2,Shubham,CSE,78,43
3,Ved,CSE,43,43
4,Vinay,ECE,95,81
5,Rohit,ECE,95,81
6,Ankit,ECE,88,81
7,Anand,ECE,81,81
8,Rishabh,EEE,91,55
9,Nitish,EEE,82,55


# Q7: Lead & LAG

In [2]:
marks = pd.read_csv('/content/W16S1_student_marks.csv')
marks.head(2)

,name,branch,marks
0,Nitish,EEE,82
1,Rishabh,EEE,91


In [ ]:
pysqldf('''
SELECT *,
LAG(marks) OVER(PARTITION BY branch ORDER BY student_id),
LEAD(marks) OVER(PARTITION BY branch ORDER BY student_id)
FROM marks


''')

,student_id,name,branch,marks,LAG(marks) OVER(PARTITION BY branch ORDER BY student_id),LEAD(marks) OVER(PARTITION BY branch ORDER BY student_id)
0,5,Shubham,CSE,78,NaN,43.0
1,6,Ved,CSE,43,78.0,98.0
2,7,Deepak,CSE,98,43.0,95.0
3,8,Arpan,CSE,95,98.0,NaN
4,9,Vinay,ECE,95,NaN,88.0
5,10,Ankit,ECE,88,95.0,81.0
6,11,Anand,ECE,81,88.0,95.0
7,12,Rohit,ECE,95,81.0,NaN
8,1,Nitish,EEE,82,NaN,91.0
9,2,Rishabh,EEE,91,82.0,69.0


# Q8: Find MoM growth for Zomato

In [8]:
pysqldf('''
SELECT strftime('%m', date),SUM(AMOUNT),
(SUM(AMOUNT) - LAG(SUM(amount)) OVER(ORDER BY strftime('%m', date))/LAG(SUM(amount)) OVER(ORDER BY strftime('%m', date))
FROM orders
GROUP BY strftime('%m', date)
ORDER BY strftime('%m', date) ASC
''')

PandaSQLException: (sqlite3.OperationalError) near "FROM": syntax error
[SQL: 
SELECT strftime('%m', date),SUM(AMOUNT),
(SUM(AMOUNT) - LAG(SUM(amount)) OVER(ORDER BY strftime('%m', date))/LAG(SUM(amount)) OVER(ORDER BY strftime('%m', date))
FROM orders
GROUP BY strftime('%m', date)
ORDER BY strftime('%m', date) ASC
]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [9]:
pysqldf("""
SELECT
    month,
    total_amount,
    (total_amount - LAG(total_amount) OVER(ORDER BY month)) * 1.0 /
        LAG(total_amount) OVER(ORDER BY month) AS growth_rate
FROM (
    SELECT
        strftime('%m', date) AS month,
        SUM(amount) AS total_amount
    FROM orders
    GROUP BY strftime('%m', date)
)
ORDER BY month
""")

,month,total_amount,growth_rate
0,05,2425,NaN
1,06,3220,0.327835
2,07,4845,0.504658
